# 隐藏层

我们成功训练了第一个神经网络模型，它可以根据天气预报来预测冰激凌的销量。

但冰激凌的销量真的是直接由天气情况决定吗？

事实上，天气并不是直接影响冰激凌销量的因素。天气首先影响的是人们的行为和偏好，例如是否愿意出门，以及是否想吃冰激凌。

* 天气凉爽时：人们更愿意出门，但未必想吃冰激凌；
* 天气酷热时：人们更想吃冰激凌，但可能不愿意出门；
* 天气寒冷时：人们既不愿意出门，也不想吃冰激凌。

从这个角度看，**冰激凌的销量**并不是由天气直接决定的，而是由人们**出门的愿望**和**吃冰激凌的愿望**共同决定的。

---

那么，我们是否可以这样做：

* 先用一个模型根据天气预测这些中间因素；
* 再用另一个模型根据这些中间因素预测冰激凌销量？

更进一步，我们是否可以把前一个模型的输出，直接作为后一个模型的输入，并将这两个模型连接成一个整体？

如果可以，那么在训练时，我们是不是可以把冰激凌销量的预测误差，通过反向传播传递回前面的模型，使两个模型能够同时被训练？

---

这种在网络模型内部自动学习中间表示的能力，正是深度神经网络的独特之处。

## 层

我们设想一下，是否可以把我们的模型分成两**层**（Layer）：

* 第一层：根据天气情况预测两个数据：人们出门的愿望，和吃冰激凌的愿望；
* 第二层：利用第一层的两个预测结果来预测冰激凌的销量。

这样的模型是否可行？能训练成功吗？

In [1]:
from abc import ABC, abstractmethod

import numpy as np

In [2]:
np.random.seed(42)

``💡 函数 np.random.seed() 用于初始化随机数，确保每次运行都产生一样的随机数，结果可复现。``

## 张量

In [3]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    def __str__(self):
        return f'Tensor({self.data})'

## 数据集

In [4]:
class Dataset:

    def __init__(self, batch_size=1):
        self.batch_size = batch_size
        self.load()
        self.train()

    def load(self):
        self.train_data = ([[22.5, 72.0],
                            [31.4, 45.0],
                            [19.8, 85.0],
                            [27.6, 63.0]],
                           [[95],
                            [210],
                            [70],
                            [155]])
        self.test_data = ([[28.1, 58.0]],
                          [[165]])

    def train(self):
        self.data = self.train_data

    def eval(self):
        self.data = self.test_data

    def all(self):
        x, y = self.data
        return Tensor(x), Tensor(y)

    def __len__(self):
        x, *_ = self.data
        return len(x) // self.batch_size

    def __getitem__(self, index):
        s = slice(index * self.batch_size, (index + 1) * self.batch_size)
        x, y = self.data
        return Tensor(x[s]), Tensor(y[s])

## 模型

In [5]:
class Layer(ABC):

    def __call__(self, x: Tensor):
        return self.forward(x)

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    @property
    def parameters(self):
        return []

## 线性层

In [6]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.random.randn(out_size, in_size) * np.sqrt(2 / in_size))
        self.bias = Tensor(np.zeros(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)
            x.grad += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

## 顺序层

In [7]:
class Sequential(Layer):

    def __init__(self, layers):
        self.layers = layers

    def forward(self, x: Tensor):
        for l in self.layers:
            x = l(x)
        return x

    @property
    def parameters(self):
        return [p for l in self.layers for p in l.parameters]

## 损失函数（均方误差）

In [8]:
class MSELoss:

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data) / y.data.size

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

## 优化器（随机梯度下降）

In [9]:
class SGDOptimizer:

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def zero_grad(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

## 训练器

In [10]:
class Trainer:

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    def train(self, dataset, epochs):
        dataset.train()

        for epoch in range(epochs):
            for i in range(len(dataset)):
                feature, label = dataset[i]

                self.optimizer.zero_grad()
                prediction = self.layer(feature)
                loss = self.loss_fn(prediction, label)
                loss.backward()
                self.optimizer.step()

    def test(self, dataset):
        dataset.eval()

        feature, label = dataset.all()
        prediction = self.layer(feature)
        loss = self.loss_fn(prediction, label)
        return prediction, loss

## 超参数

### 学习率

In [11]:
LEARNING_RATE = 0.00001

### 批大小

In [12]:
BATCH_SIZE = 2

### 轮数

In [13]:
EPOCHS = 1000

## 建模

In [14]:
dataset = Dataset(BATCH_SIZE)
model = Sequential([
    Linear(2, 4),
    Linear(4, 1),
])
loss_fn = MSELoss()
optimizer = SGDOptimizer(model.parameters, lr=LEARNING_RATE)
trainer = Trainer(model, loss_fn, optimizer)

## 训练

In [15]:
trainer.train(dataset, EPOCHS)

## 评估

In [16]:
prediction, loss = trainer.test(dataset)
print(f'prediction:\t{prediction}\nloss:\t{loss}')

prediction:	Tensor([[166.52357679]])
loss:	Tensor(2.321286244423445)
